# Advanced Pull Pipelines in Python — Problems with Complete Solutions

This notebook develops production-quality **pull pipelines** with iterators and generators.

```text
consumer <-- transformation <-- transformation <-- source
```

The consumer asks for one item at a time, and each upstream stage performs only enough work to produce that item.

## Learning objectives

- Build lazy, resource-safe CSV sources.
- Convert raw strings into validated typed records.
- Compose reusable map, filter, inspect, batching, and window stages.
- Prove laziness and short-circuiting.
- Compute bounded-memory top-k and online statistics.
- Handle malformed rows without losing the entire stream.
- Understand single-pass iterators and `itertools.tee` buffering.
- Build an ergonomic pipeline API with `|`.
- Test behavior, not only final values.

Every numbered problem includes a complete solution and executable checks.

## 0. Setup — create a self-contained `cars.csv`

The original lesson expects `cars.csv`. This notebook creates a compact sample only when that file does not already exist.

Schema:

```text
name, mpg, cylinders, displacement, horsepower, weight,
acceleration, model_year, origin
```

In [2]:
from __future__ import annotations

from pathlib import Path
from typing import Any, Callable, Generic, Iterable, Iterator, Sequence, TypeVar
from dataclasses import asdict, dataclass
from collections import defaultdict, deque
import csv
import heapq
import itertools
import math
import operator
import statistics
import tempfile

DATA_FILE = Path("cars.csv")

SAMPLE_CSV = """name,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin
Chevrolet Chevelle Malibu,18.0,8,307.0,130.0,3504,12.0,70,US
Buick Skylark 320,15.0,8,350.0,165.0,3693,11.5,70,US
Plymouth Satellite,18.0,8,318.0,150.0,3436,11.0,70,US
AMC Rebel SST,16.0,8,304.0,150.0,3433,12.0,70,US
Ford Torino,17.0,8,302.0,140.0,3449,10.5,70,US
Chevrolet Impala,14.0,8,454.0,220.0,4354,9.0,70,US
Chevrolet Chevelle Concours (sw),,8,350.0,165.0,4142,11.5,70,US
Chevrolet Monte Carlo,15.0,8,400.0,150.0,3761,9.5,70,US
Chevrolet Vega 2300,28.0,4,140.0,90.0,2264,15.5,71,US
Toyota Corolla,31.0,4,71.0,65.0,1773,19.0,71,Japan
Datsun 510,27.0,4,97.0,88.0,2130,14.5,71,Japan
Volkswagen 1131 Deluxe Sedan,26.0,4,97.0,46.0,1835,20.5,70,Europe
Peugeot 504,25.0,4,110.0,87.0,2672,17.5,70,Europe
Audi 100 LS,24.0,4,107.0,90.0,2430,14.5,70,Europe
Saab 99e,25.0,4,104.0,95.0,2375,17.5,70,Europe
BMW 2002,26.0,4,121.0,113.0,2234,12.5,70,Europe
Chevrolet Monte Carlo S,15.0,8,350.0,145.0,4082,13.0,73,US
Chevrolet Monte Carlo Landau,15.5,8,350.0,170.0,4165,11.4,77,US
Chevrolet Monte Carlo Landau,19.2,8,305.0,145.0,3425,13.2,78,US
Honda Civic,33.0,4,91.0,53.0,1795,17.4,76,Japan
Honda Accord LX,29.5,4,98.0,68.0,2135,16.6,78,Japan
Toyota Celica GT,32.0,4,144.0,96.0,2665,13.9,82,Japan
Volvo 244DL,22.0,4,121.0,98.0,2945,14.5,75,Europe
Mercedes-Benz 240d,30.0,4,146.0,67.0,3250,21.8,80,Europe
Ford Pinto,25.0,4,98.0,,2046,19.0,71,US
AMC Gremlin,21.0,6,199.0,90.0,2648,15.0,70,US
Mazda RX-3,18.0,3,70.0,90.0,2124,13.5,73,Japan
Renault 12,26.0,4,96.0,69.0,2189,18.0,72,Europe
Dodge Colt,28.0,4,90.0,75.0,2125,14.5,74,US
Subaru GL,33.8,4,97.0,67.0,2145,18.0,80,Japan
"""

if not DATA_FILE.exists():
    DATA_FILE.write_text(SAMPLE_CSV, encoding="utf-8")

# print(f"Using: {DATA_FILE.resolve()}")
print(f"Data rows: {len(DATA_FILE.read_text(encoding='utf-8').splitlines()) - 1}")

Data rows: 406


## Problem 1 — Build a lazy, resource-safe CSV source

Requirements:

1. Use `newline=""` and an explicit encoding.
2. Address columns by name with `csv.DictReader`.
3. Detect comma-, semicolon-, tab-, or pipe-delimited files.
4. Normalize common header variants at the source boundary.
5. Accept both course-style headers such as `Car`, `MPG`, and `Model` and snake-case headers such as `name`, `mpg`, and `model_year`.
6. Validate the schema before parsing data rows.
7. Open the file only when iteration begins.
8. Close it when iteration finishes or the generator is closed.


In [3]:
CANONICAL_COLUMNS = {
    "name",
    "mpg",
    "cylinders",
    "displacement",
    "horsepower",
    "weight",
    "acceleration",
    "model_year",
    "origin",
}

HEADER_ALIASES = {
    # Course dataset headers
    "car": "name",
    "model": "model_year",

    # Canonical and common variants
    "name": "name",
    "mpg": "mpg",
    "cylinders": "cylinders",
    "cylinder": "cylinders",
    "displacement": "displacement",
    "horsepower": "horsepower",
    "horse_power": "horsepower",
    "weight": "weight",
    "acceleration": "acceleration",
    "model_year": "model_year",
    "modelyear": "model_year",
    "year": "model_year",
    "origin": "origin",
}


def normalize_header(header: str | None) -> str:
    """Normalize capitalization, punctuation, whitespace, and a UTF-8 BOM."""
    if header is None:
        return ""

    cleaned = header.removeprefix("\ufeff").strip().casefold()
    normalized = "".join(
        character if character.isalnum() else "_"
        for character in cleaned
    )
    return "_".join(part for part in normalized.split("_") if part)


def canonicalize_headers(fieldnames: list[str | None]) -> list[str]:
    """Map raw CSV headers to the notebook's canonical schema."""
    canonical: list[str] = []

    for raw_header in fieldnames:
        normalized = normalize_header(raw_header)
        canonical_name = HEADER_ALIASES.get(normalized, normalized)

        if not canonical_name:
            raise ValueError(f"CSV contains an empty header: {fieldnames!r}")
        if canonical_name in canonical:
            raise ValueError(
                "CSV headers become duplicates after normalization: "
                f"{fieldnames!r}"
            )

        canonical.append(canonical_name)

    missing = CANONICAL_COLUMNS - set(canonical)
    if missing:
        raise ValueError(
            "CSV schema is missing required columns "
            f"{sorted(missing)}. Detected headers: {fieldnames!r}; "
            f"normalized headers: {canonical!r}"
        )

    return canonical


def sniff_dialect(file_obj, *, sample_size: int = 4096):
    """Return a detected CSV dialect without changing the stream position."""
    start = file_obj.tell()
    sample = file_obj.read(sample_size)
    file_obj.seek(start)

    try:
        return csv.Sniffer().sniff(sample, delimiters=",;\t|")
    except csv.Error:
        first_line = sample.splitlines()[0] if sample else ""
        delimiter = ";" if first_line.count(";") > first_line.count(",") else ","

        class FallbackDialect(csv.excel):
            pass

        FallbackDialect.delimiter = delimiter
        return FallbackDialect


def iter_csv_dicts(
    file_name: str | Path,
    *,
    encoding: str = "utf-8",
) -> Iterator[dict[str, str]]:
    """
    Lazily yield rows using canonical column names.

    Accepted examples:
        Car;MPG;Cylinders;...;Model;Origin
        name,mpg,cylinders,...,model_year,origin
    """
    with Path(file_name).open("r", encoding=encoding, newline="") as file_obj:
        dialect = sniff_dialect(file_obj)
        reader = csv.DictReader(file_obj, dialect=dialect)

        if reader.fieldnames is None:
            raise ValueError("CSV file has no header row")

        raw_fieldnames = list(reader.fieldnames)
        reader.fieldnames = canonicalize_headers(raw_fieldnames)

        for row_number, row in enumerate(reader, start=2):
            if None in row:
                raise ValueError(
                    f"row {row_number} contains more values than headers: "
                    f"{row[None]!r}"
                )
            yield row


rows = iter_csv_dicts(DATA_FILE)
print(type(rows))
print(next(rows))
rows.close()


<class 'generator'>
{'name': 'Chevrolet Chevelle Malibu', 'mpg': '18.0', 'cylinders': '8', 'displacement': '307.0', 'horsepower': '130.0', 'weight': '3504.', 'acceleration': '12.0', 'model_year': '70', 'origin': 'US'}


Calling `iter_csv_dicts(...)` only creates a generator. The file is opened on the first `next()` call, so the downstream consumer controls demand.

Header normalization is performed once, immediately after reading the header row. Every later stage receives this stable schema:

```text
name, mpg, cylinders, displacement, horsepower,
weight, acceleration, model_year, origin
```

For the supplied course file, the important aliases are:

```text
Car   -> name
Model -> model_year
```


## Problem 2 — Parse raw rows into validated typed records

Create a frozen, slotted `Car` dataclass. Convert numeric columns, allow missing `mpg` and `horsepower`, reject invalid required fields, and include the source row number in every parsing error.

In [4]:
MISSING_MARKERS = {"", "?", "na", "n/a", "null", "none"}


@dataclass(frozen=True, slots=True)
class Car:
    name: str
    mpg: float | None
    cylinders: int
    displacement: float
    horsepower: float | None
    weight: float
    acceleration: float
    model_year: int
    origin: str


class RowParseError(ValueError):
    def __init__(self, row_number: int, field: str, value: Any, message: str):
        self.row_number = row_number
        self.field = field
        self.value = value
        super().__init__(
            f"row {row_number}, field {field!r}, value {value!r}: {message}"
        )


def optional_float(value: str | None, *, row_number: int, field: str):
    normalized = "" if value is None else value.strip()
    if normalized.casefold() in MISSING_MARKERS:
        return None
    try:
        return float(normalized)
    except ValueError as exc:
        raise RowParseError(
            row_number, field, value, "expected a floating-point number"
        ) from exc


def required_float(value: str | None, *, row_number: int, field: str) -> float:
    parsed = optional_float(value, row_number=row_number, field=field)
    if parsed is None:
        raise RowParseError(row_number, field, value, "value is required")
    return parsed


def required_int(value: str | None, *, row_number: int, field: str) -> int:
    normalized = "" if value is None else value.strip()
    if normalized.casefold() in MISSING_MARKERS:
        raise RowParseError(row_number, field, value, "value is required")
    try:
        return int(normalized)
    except ValueError as exc:
        raise RowParseError(row_number, field, value, "expected an integer") from exc


def canonicalize_row_keys(row: dict[str, str]) -> dict[str, str]:
    """Return a new row with canonical, case-insensitive column names.

    This makes ``parse_car`` robust even when it receives a raw ``DictReader``
    row whose keys are ``Car``, ``MPG``, and ``Model``. It therefore does not
    depend on a previously executed notebook cell having replaced the reader's
    field names.
    """
    aliases = {
        "car": "name",
        "name": "name",
        "mpg": "mpg",
        "cylinder": "cylinders",
        "cylinders": "cylinders",
        "displacement": "displacement",
        "horsepower": "horsepower",
        "horse_power": "horsepower",
        "weight": "weight",
        "acceleration": "acceleration",
        "model": "model_year",
        "model_year": "model_year",
        "modelyear": "model_year",
        "year": "model_year",
        "origin": "origin",
    }

    normalized_row: dict[str, str] = {}
    for raw_key, value in row.items():
        if raw_key is None:
            continue
        cleaned = raw_key.removeprefix("\ufeff").strip().casefold()
        normalized = "".join(
            character if character.isalnum() else "_"
            for character in cleaned
        )
        normalized = "_".join(
            part for part in normalized.split("_") if part
        )
        canonical_key = aliases.get(normalized, normalized)
        normalized_row[canonical_key] = value

    return normalized_row


def parse_car(row: dict[str, str], row_number: int) -> Car:
    row = canonicalize_row_keys(row)
    name = (row.get("name") or "").strip()
    origin = (row.get("origin") or "").strip()
    if not name:
        raise RowParseError(row_number, "name", row.get("name"), "value is required")
    if not origin:
        raise RowParseError(
            row_number, "origin", row.get("origin"), "value is required"
        )

    car = Car(
        name=name,
        mpg=optional_float(row.get("mpg"), row_number=row_number, field="mpg"),
        cylinders=required_int(
            row.get("cylinders"), row_number=row_number, field="cylinders"
        ),
        displacement=required_float(
            row.get("displacement"), row_number=row_number, field="displacement"
        ),
        horsepower=optional_float(
            row.get("horsepower"), row_number=row_number, field="horsepower"
        ),
        weight=required_float(
            row.get("weight"), row_number=row_number, field="weight"
        ),
        acceleration=required_float(
            row.get("acceleration"), row_number=row_number, field="acceleration"
        ),
        model_year=required_int(
            row.get("model_year"), row_number=row_number, field="model_year"
        ),
        origin=origin,
    )

    if car.cylinders <= 0:
        raise RowParseError(
            row_number, "cylinders", car.cylinders, "must be positive"
        )
    if car.weight <= 0:
        raise RowParseError(row_number, "weight", car.weight, "must be positive")
    return car


def iter_cars(file_name: str | Path) -> Iterator[Car]:
    for row_number, raw_row in enumerate(iter_csv_dicts(file_name), start=2):
        yield parse_car(raw_row, row_number)


for car in itertools.islice(iter_cars(DATA_FILE), 3):
    print(car)

Car(name='Chevrolet Chevelle Malibu', mpg=18.0, cylinders=8, displacement=307.0, horsepower=130.0, weight=3504.0, acceleration=12.0, model_year=70, origin='US')
Car(name='Buick Skylark 320', mpg=15.0, cylinders=8, displacement=350.0, horsepower=165.0, weight=3693.0, acceleration=11.5, model_year=70, origin='US')
Car(name='Plymouth Satellite', mpg=18.0, cylinders=8, displacement=318.0, horsepower=150.0, weight=3436.0, acceleration=11.0, model_year=70, origin='US')


In [5]:
# Regression test for the exact course-style headers and semicolon delimiter.
raw_course_row = {
    "Car": "Chevrolet Chevelle Malibu",
    "MPG": "18.0",
    "Cylinders": "8",
    "Displacement": "307.0",
    "Horsepower": "130.0",
    "Weight": "3504.",
    "Acceleration": "12.0",
    "Model": "70",
    "Origin": "US",
}

parsed_course_row = parse_car(raw_course_row, row_number=2)
assert parsed_course_row.name == "Chevrolet Chevelle Malibu"
assert parsed_course_row.model_year == 70
print(parsed_course_row)


Car(name='Chevrolet Chevelle Malibu', mpg=18.0, cylinders=8, displacement=307.0, horsepower=130.0, weight=3504.0, acceleration=12.0, model_year=70, origin='US')


## Problem 3 — Implement reusable lazy stages

Create generic lazy equivalents of `map`, `filter`, `take`, `drop`, and `inspect`. The inspection stage performs a side effect while yielding each original item unchanged.

In [6]:
T = TypeVar("T")
U = TypeVar("U")


def map_items(function: Callable[[T], U], items: Iterable[T]) -> Iterator[U]:
    for item in items:
        yield function(item)


def filter_items(
    predicate: Callable[[T], bool],
    items: Iterable[T],
) -> Iterator[T]:
    for item in items:
        if predicate(item):
            yield item


def take(count: int, items: Iterable[T]) -> Iterator[T]:
    if count < 0:
        raise ValueError("count must be non-negative")
    yield from itertools.islice(items, count)


def drop(count: int, items: Iterable[T]) -> Iterator[T]:
    if count < 0:
        raise ValueError("count must be non-negative")
    yield from itertools.islice(items, count, None)


def inspect_items(
    action: Callable[[T], Any],
    items: Iterable[T],
) -> Iterator[T]:
    for item in items:
        action(item)
        yield item


result = take(
    5,
    map_items(
        lambda car: (car.name, car.mpg),
        filter_items(
            lambda car: car.origin == "Japan" and car.mpg is not None,
            iter_cars(DATA_FILE),
        ),
    ),
)

list(result)

[('Toyota Corolla Mark ii', 24.0),
 ('Datsun PL510', 27.0),
 ('Datsun PL510', 27.0),
 ('Toyota Corolla', 25.0),
 ('Toyota Corolla 1200', 31.0)]

## Problem 4 — Prove short-circuiting with instrumentation

Take three squared even numbers from a source of 100 integers. Record every value produced upstream and prove that the whole source was not consumed.

In [7]:
def traced_count(stop: int, events: list[str]) -> Iterator[int]:
    events.append("source opened")
    try:
        for value in range(stop):
            events.append(f"produced {value}")
            yield value
    finally:
        events.append("source closed")


events: list[str] = []
answer = list(
    take(
        3,
        map_items(
            lambda value: value * value,
            filter_items(lambda value: value % 2 == 0, traced_count(100, events)),
        ),
    )
)

print("answer:", answer)
print("events:", events)
assert answer == [0, 4, 16]
assert "produced 5" not in events

answer: [0, 4, 16]
events: ['source opened', 'produced 0', 'produced 1', 'produced 2', 'produced 3', 'produced 4', 'source closed']


A filter may pull several upstream values to emit one downstream value. Here the source produces `0` through `4`, enough to find three even values.

## Problem 5 — Build configurable text and numeric predicates

Implement:

- case-insensitive name matching with `all`/`any` modes;
- a reusable numeric range predicate;
- a pipeline for US cars whose names contain both `Chevrolet` and `Carlo`, with MPG at least 15.

In [8]:
def name_contains(
    *words: str,
    mode: str = "all",
    case_sensitive: bool = False,
) -> Callable[[Car], bool]:
    if mode not in {"all", "any"}:
        raise ValueError("mode must be 'all' or 'any'")

    normalized = words if case_sensitive else tuple(word.casefold() for word in words)

    def predicate(car: Car) -> bool:
        candidate = car.name if case_sensitive else car.name.casefold()
        checks = (word in candidate for word in normalized)
        return all(checks) if mode == "all" else any(checks)

    return predicate


def between(
    attribute: str,
    *,
    minimum: float | None = None,
    maximum: float | None = None,
    include_minimum: bool = True,
    include_maximum: bool = True,
) -> Callable[[Any], bool]:
    def predicate(item: Any) -> bool:
        value = getattr(item, attribute)
        if value is None:
            return False
        if minimum is not None:
            if include_minimum and value < minimum:
                return False
            if not include_minimum and value <= minimum:
                return False
        if maximum is not None:
            if include_maximum and value > maximum:
                return False
            if not include_maximum and value >= maximum:
                return False
        return True
    return predicate


chevrolet_carlos = filter_items(
    between("mpg", minimum=15),
    filter_items(
        lambda car: car.origin == "US",
        filter_items(name_contains("Chevrolet", "Carlo"), iter_cars(DATA_FILE)),
    ),
)

for car in chevrolet_carlos:
    print(car)

Car(name='Chevrolet Monte Carlo', mpg=15.0, cylinders=8, displacement=400.0, horsepower=150.0, weight=3761.0, acceleration=9.5, model_year=70, origin='US')
Car(name='Chevrolet Monte Carlo S', mpg=15.0, cylinders=8, displacement=350.0, horsepower=145.0, weight=4082.0, acceleration=13.0, model_year=73, origin='US')
Car(name='Chevrolet Monte Carlo Landau', mpg=15.5, cylinders=8, displacement=350.0, horsepower=170.0, weight=4165.0, acceleration=11.4, model_year=77, origin='US')
Car(name='Chevrolet Monte Carlo Landau', mpg=19.2, cylinders=8, displacement=305.0, horsepower=145.0, weight=3425.0, acceleration=13.2, model_year=78, origin='US')


## Problem 6 — Compose stage factories left to right

Nested function calls become difficult to scan. Implement stage factories and `compose` so configuration appears in data-flow order.

In [9]:
Stage = Callable[[Iterable[Any]], Iterable[Any]]


def mapping(function: Callable[[T], U]):
    return lambda items: map_items(function, items)


def filtering(predicate: Callable[[T], bool]):
    return lambda items: filter_items(predicate, items)


def taking(count: int):
    return lambda items: take(count, items)


def inspecting(action: Callable[[T], Any]):
    return lambda items: inspect_items(action, items)


def compose(*stages: Stage) -> Stage:
    def pipeline(items: Iterable[Any]) -> Iterable[Any]:
        current = items
        for stage in stages:
            current = stage(current)
        return current
    return pipeline


japanese_summary = compose(
    filtering(lambda car: car.origin == "Japan"),
    filtering(lambda car: car.mpg is not None),
    filtering(between("mpg", minimum=28)),
    mapping(lambda car: {"name": car.name, "mpg": car.mpg}),
    taking(6),
)

list(japanese_summary(iter_cars(DATA_FILE)))

[{'name': 'Toyota Corolla 1200', 'mpg': 31.0},
 {'name': 'Datsun 1200', 'mpg': 35.0},
 {'name': 'Datsun 510 (sw)', 'mpg': 28.0},
 {'name': 'Datsun B210', 'mpg': 31.0},
 {'name': 'Toyota Corolla 1200', 'mpg': 32.0},
 {'name': 'Toyota Corolla', 'mpg': 31.0}]

## Problem 7 — Process an arbitrary stream in bounded-memory batches

Write `batched(items, size)` that yields tuples of at most `size` items, rejects invalid sizes, and works with infinite inputs.

In [10]:
def batched(items: Iterable[T], size: int) -> Iterator[tuple[T, ...]]:
    if size <= 0:
        raise ValueError("size must be positive")
    iterator = iter(items)
    while batch := tuple(itertools.islice(iterator, size)):
        yield batch


valid_mpg = filter_items(lambda car: car.mpg is not None, iter_cars(DATA_FILE))

summaries = map_items(
    lambda batch: {
        "count": len(batch),
        "first": batch[0].name,
        "last": batch[-1].name,
        "average_mpg": statistics.fmean(
            car.mpg for car in batch if car.mpg is not None
        ),
    },
    batched(valid_mpg, 5),
)

for summary in summaries:
    print(summary)

{'count': 5, 'first': 'Chevrolet Chevelle Malibu', 'last': 'Ford Torino', 'average_mpg': 16.8}
{'count': 5, 'first': 'Ford Galaxie 500', 'last': 'AMC Ambassador DPL', 'average_mpg': 14.4}
{'count': 5, 'first': 'Citroen DS-21 Pallas', 'last': 'AMC Rebel SST (sw)', 'average_mpg': 0.0}
{'count': 5, 'first': 'Dodge Challenger SE', 'last': 'Buick Estate Wagon (sw)', 'average_mpg': 11.6}
{'count': 5, 'first': 'Toyota Corolla Mark ii', 'last': 'Datsun PL510', 'average_mpg': 22.4}
{'count': 5, 'first': 'Volkswagen 1131 Deluxe Sedan', 'last': 'BMW 2002', 'average_mpg': 25.2}
{'count': 5, 'first': 'AMC Gremlin', 'last': 'Hi 1200D', 'average_mpg': 12.2}
{'count': 5, 'first': 'Datsun PL510', 'last': 'Volkswagen Super Beetle 117', 'average_mpg': 21.0}
{'count': 5, 'first': 'AMC Gremlin', 'last': 'AMC Matador', 'average_mpg': 17.8}
{'count': 5, 'first': 'Chevrolet Impala', 'last': 'Dodge Monaco (sw)', 'average_mpg': 13.6}
{'count': 5, 'first': 'Ford Country Squire (sw)', 'last': 'Pontiac Firebird', 

## Problem 8 — Build a fixed-memory sliding window

Use a `deque(maxlen=size)` to produce overlapping windows. Find adjacent valid-MPG cars where MPG improves by at least 8.

In [11]:
def sliding_window(items: Iterable[T], size: int) -> Iterator[tuple[T, ...]]:
    if size <= 0:
        raise ValueError("size must be positive")
    window: deque[T] = deque(maxlen=size)
    for item in items:
        window.append(item)
        if len(window) == size:
            yield tuple(window)


valid_mpg = filter_items(lambda car: car.mpg is not None, iter_cars(DATA_FILE))
large_improvements = filter_items(
    lambda pair: pair[1].mpg - pair[0].mpg >= 8,
    sliding_window(valid_mpg, 2),
)

for previous, current in large_improvements:
    print(f"{previous.name}: {previous.mpg} -> {current.name}: {current.mpg}")

AMC Rebel SST (sw): 0.0 -> Dodge Challenger SE: 15.0
Ford Mustang Boss 302: 0.0 -> Chevrolet Monte Carlo: 15.0
Buick Estate Wagon (sw): 14.0 -> Toyota Corolla Mark ii: 24.0
Hi 1200D: 9.0 -> Datsun PL510: 27.0
Volkswagen Super Beetle 117: 0.0 -> AMC Gremlin: 19.0
Pontiac Grand Prix: 16.0 -> Fiat 128: 29.0
Dodge Dart Custom: 15.0 -> Saab 99le: 24.0
Oldsmobile Omega: 11.0 -> Plymouth Duster: 20.0
Chevrolet Nova: 15.0 -> Datsun B210: 31.0
AMC Matador (sw): 14.0 -> Audi Fox: 29.0
Ford Mustang II: 13.0 -> Toyota Corolla: 29.0
Ford Pinto: 18.0 -> Volkswagen Rabbit: 29.0
Saab 99LE: 25.0 -> Honda Civic CVCC: 33.0
AMC Pacer d/l: 17.5 -> Volkswagen Rabbit: 29.5
Dodge D100: 13.0 -> Honda Accord CVCC: 31.5
Plymouth Arrow GS: 25.5 -> Datsun F-10 Hatchback: 33.5
Ford Thunderbird: 16.0 -> Volkswagen Rabbit Custom: 29.0
Mazda RX-4: 21.5 -> Volkswagen Rabbit Custom Diesel: 43.1
Dodge Magnum XE: 17.5 -> Chevrolet Chevette: 30.0
Peugeot 604sl: 16.2 -> Volkswagen Scirocco: 31.5
Chrysler Lebaron Town @ Coun

## Problem 9 — Compute online statistics with Welford's algorithm

Track count, mean, sample variance, standard deviation, minimum, and maximum without storing all values. Then compute MPG statistics by origin in one pass.

In [12]:
@dataclass
class OnlineStats:
    count: int = 0
    mean: float = 0.0
    m2: float = 0.0
    minimum: float = math.inf
    maximum: float = -math.inf

    def update(self, value: float) -> None:
        self.count += 1
        delta = value - self.mean
        self.mean += delta / self.count
        delta2 = value - self.mean
        self.m2 += delta * delta2
        self.minimum = min(self.minimum, value)
        self.maximum = max(self.maximum, value)

    @property
    def sample_variance(self) -> float:
        return math.nan if self.count < 2 else self.m2 / (self.count - 1)

    @property
    def sample_stddev(self) -> float:
        return math.sqrt(self.sample_variance)

    def as_dict(self) -> dict[str, float | int]:
        return {
            "count": self.count,
            "mean": self.mean,
            "sample_variance": self.sample_variance,
            "sample_stddev": self.sample_stddev,
            "minimum": self.minimum,
            "maximum": self.maximum,
        }


def mpg_stats_by_origin(cars: Iterable[Car]) -> dict[str, OnlineStats]:
    groups: dict[str, OnlineStats] = defaultdict(OnlineStats)
    for car in cars:
        if car.mpg is not None:
            groups[car.origin].update(car.mpg)
    return dict(groups)


origin_stats = mpg_stats_by_origin(iter_cars(DATA_FILE))
for origin, stats in sorted(origin_stats.items()):
    print(origin, stats.as_dict())

Europe {'count': 73, 'mean': 26.745205479452057, 'sample_variance': 74.40917808219177, 'sample_stddev': 8.626075473944786, 'minimum': 0.0, 'maximum': 44.3}
Japan {'count': 79, 'mean': 30.45063291139241, 'sample_variance': 37.088685491723446, 'sample_stddev': 6.09004806973832, 'minimum': 18.0, 'maximum': 46.6}
US {'count': 254, 'mean': 19.688188976377937, 'sample_variance': 48.001203821854325, 'sample_stddev': 6.928290108089754, 'minimum': 0.0, 'maximum': 39.0}


### Verification against a materialized reference

In [13]:
reference: dict[str, list[float]] = defaultdict(list)
for car in iter_cars(DATA_FILE):
    if car.mpg is not None:
        reference[car.origin].append(car.mpg)

for origin, values in reference.items():
    assert math.isclose(origin_stats[origin].mean, statistics.fmean(values))
    if len(values) >= 2:
        assert math.isclose(
            origin_stats[origin].sample_variance,
            statistics.variance(values),
        )

print("Online calculations match statistics module references.")

Online calculations match statistics module references.


## Problem 10 — Find top-k records with bounded memory

Avoid sorting the entire stream. Use a heap-based algorithm to find the five highest-MPG cars.

In [14]:
def top_k(
    items: Iterable[T],
    k: int,
    *,
    key: Callable[[T], Any],
) -> list[T]:
    if k < 0:
        raise ValueError("k must be non-negative")
    return heapq.nlargest(k, items, key=key)


top_five = top_k(
    filter_items(lambda car: car.mpg is not None, iter_cars(DATA_FILE)),
    5,
    key=lambda car: car.mpg,
)

for rank, car in enumerate(top_five, start=1):
    print(f"{rank}. {car.name}: {car.mpg} MPG")

1. Mazda GLC: 46.6 MPG
2. Honda Civic 1500 gl: 44.6 MPG
3. Volkswagen Rabbit C (Diesel): 44.3 MPG
4. Volkswagen Pickup: 44.0 MPG
5. Volkswagen Dasher (diesel): 43.4 MPG


## Problem 11 — Branch a single-pass stream safely

An iterator is normally consumed once. First use `itertools.tee` to make three branches, then implement the preferred single-pass count aggregation.

`tee` may buffer many items when consumers advance at different speeds.

In [15]:
us_stream, japan_stream, europe_stream = itertools.tee(iter_cars(DATA_FILE), 3)

counts_with_tee = {
    "US": sum(car.origin == "US" for car in us_stream),
    "Japan": sum(car.origin == "Japan" for car in japan_stream),
    "Europe": sum(car.origin == "Europe" for car in europe_stream),
}

counts_with_tee

{'US': 254, 'Japan': 79, 'Europe': 73}

In [16]:
def count_by(items: Iterable[T], key: Callable[[T], U]) -> dict[U, int]:
    counts: dict[U, int] = defaultdict(int)
    for item in items:
        counts[key(item)] += 1
    return dict(counts)


counts_single_pass = count_by(iter_cars(DATA_FILE), key=lambda car: car.origin)

# `count_by` omits absent keys, while the tee example explicitly stores zeroes.
# Normalize both views before comparing them so the test works with any subset
# of origins, including a small teaching sample containing only US cars.
all_origins = set(counts_with_tee) | set(counts_single_pass)
assert {
    origin: counts_single_pass.get(origin, 0)
    for origin in all_origins
} == {
    origin: counts_with_tee.get(origin, 0)
    for origin in all_origins
}

counts_single_pass


{'US': 254, 'Europe': 73, 'Japan': 79}

## Problem 12 — Continue after malformed rows with a dead-letter collection

A strict parser stops at the first invalid record. Implement a policy that yields valid cars, stores structured rejected rows, and continues.

In [17]:
@dataclass(frozen=True, slots=True)
class RejectedRow:
    row_number: int
    raw_row: dict[str, str]
    error: str


def parse_cars_with_errors(
    raw_rows: Iterable[dict[str, str]],
    *,
    starting_row_number: int = 2,
) -> tuple[Iterator[Car], list[RejectedRow]]:
    rejected: list[RejectedRow] = []

    def valid_records() -> Iterator[Car]:
        for row_number, raw_row in enumerate(raw_rows, start=starting_row_number):
            try:
                yield parse_car(raw_row, row_number)
            except RowParseError as exc:
                rejected.append(
                    RejectedRow(row_number, dict(raw_row), str(exc))
                )

    return valid_records(), rejected


bad_csv = """name,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin
Valid Car,30,4,100,80,2100,15,80,US
Bad Cylinders,25,not-an-int,100,80,2200,15,80,US
Bad Weight,20,4,100,80,-1,15,80,US
,22,4,100,80,2200,15,80,US
"""

with tempfile.TemporaryDirectory() as temp_dir:
    test_file = Path(temp_dir) / "bad_cars.csv"
    test_file.write_text(bad_csv, encoding="utf-8")
    valid_iter, rejected = parse_cars_with_errors(iter_csv_dicts(test_file))
    valid = list(valid_iter)

print("valid:", valid)
print("rejected:")
for item in rejected:
    print(item)

assert len(valid) == 1
assert len(rejected) == 3

valid: [Car(name='Valid Car', mpg=30.0, cylinders=4, displacement=100.0, horsepower=80.0, weight=2100.0, acceleration=15.0, model_year=80, origin='US')]
rejected:
RejectedRow(row_number=3, raw_row={'name': 'Bad Cylinders', 'mpg': '25', 'cylinders': 'not-an-int', 'displacement': '100', 'horsepower': '80', 'weight': '2200', 'acceleration': '15', 'model_year': '80', 'origin': 'US'}, error="row 3, field 'cylinders', value 'not-an-int': expected an integer")
RejectedRow(row_number=4, raw_row={'name': 'Bad Weight', 'mpg': '20', 'cylinders': '4', 'displacement': '100', 'horsepower': '80', 'weight': '-1', 'acceleration': '15', 'model_year': '80', 'origin': 'US'}, error="row 4, field 'weight', value -1.0: must be positive")
RejectedRow(row_number=5, raw_row={'name': '', 'mpg': '22', 'cylinders': '4', 'displacement': '100', 'horsepower': '80', 'weight': '2200', 'acceleration': '15', 'model_year': '80', 'origin': 'US'}, error="row 5, field 'name', value '': value is required")


## Problem 13 — Add metrics without changing pipeline values

Count total records, missing MPG values, and records by origin. Place the inspection stage before filtering so it observes every source record.

In [18]:
@dataclass
class PipelineMetrics:
    total: int = 0
    missing_mpg: int = 0
    by_origin: dict[str, int] | None = None

    def __post_init__(self) -> None:
        if self.by_origin is None:
            self.by_origin = defaultdict(int)

    def observe(self, car: Car) -> None:
        self.total += 1
        if car.mpg is None:
            self.missing_mpg += 1
        self.by_origin[car.origin] += 1


metrics = PipelineMetrics()
high_mpg_names = list(
    compose(
        inspecting(metrics.observe),
        filtering(lambda car: car.mpg is not None and car.mpg >= 28),
        mapping(operator.attrgetter("name")),
    )(iter_cars(DATA_FILE))
)

print(high_mpg_names)
print(metrics)

['Chevrolet Vega 2300', 'Opel 1900', 'Peugeot 304', 'Fiat 124B', 'Toyota Corolla 1200', 'Datsun 1200', 'Datsun 510 (sw)', 'Dodge Colt (sw)', 'Fiat 128', 'Datsun B210', 'Toyota Corolla 1200', 'Audi Fox', 'Toyota Corolla', 'Datsun 710', 'Dodge Colt', 'Fiat x1.9', 'Toyota Corolla', 'Volkswagen Rabbit', 'Honda Civic CVCC', 'Fiat 131', 'Chevrolet Chevette', 'Volkswagen Rabbit', 'Honda Civic', 'Volkswagen Rabbit', 'Datsun B-210', 'Toyota Corolla', 'Honda Accord CVCC', 'Buick Opel Isuzu Deluxe', 'Renault 5 GTL', 'Datsun F-10 Hatchback', 'Volkswagen Rabbit Custom', 'Chevrolet Chevette', 'Dodge Colt m/m', 'Subaru DL', 'Volkswagen Dasher', 'Volkswagen Rabbit Custom Diesel', 'Ford Fiesta', 'Mazda GLC Deluxe', 'Datsun B210 GX', 'Honda Civic CVCC', 'Chevrolet Chevette', 'Dodge Omni', 'Volkswagen Scirocco', 'Honda Accord LX', 'Volkswagen Rabbit Custom', 'Mazda GLC Deluxe', 'Dodge Colt Hatchback Custom', 'Plymouth Horizon', 'Plymouth Horizon TC3', 'Datsun 210', 'Fiat Strada Custom', 'Buick Skylark Li

## Problem 14 — Build an ergonomic `Pipeline` class

Support this lazy syntax:

```python
Pipeline(source) | filtering(...) | mapping(...) | taking(...)
```

In [19]:
@dataclass
class Pipeline(Generic[T]):
    iterable: Iterable[T]

    def __iter__(self) -> Iterator[T]:
        return iter(self.iterable)

    def __or__(
        self,
        stage: Callable[[Iterable[T]], Iterable[U]],
    ) -> "Pipeline[U]":
        return Pipeline(stage(self.iterable))

    def collect(self) -> list[T]:
        return list(self)

    def first(self, default: U | None = None) -> T | U | None:
        return next(iter(self), default)


pipeline = (
    Pipeline(iter_cars(DATA_FILE))
    | filtering(lambda car: car.origin == "Europe")
    | filtering(lambda car: car.mpg is not None and car.mpg >= 25)
    | mapping(lambda car: (car.name, car.mpg))
    | taking(5)
)

pipeline.collect()

[('Volkswagen 1131 Deluxe Sedan', 26.0),
 ('Peugeot 504', 25.0),
 ('Saab 99e', 25.0),
 ('BMW 2002', 26.0),
 ('Opel 1900', 28.0)]

## Problem 15 — Compile safe declarative filter rules

Compile configuration such as:

```python
{"field": "mpg", "op": "ge", "value": 28}
```

Use an explicit operation allowlist. Never use `eval`.

In [20]:
OPERATORS: dict[str, Callable[[Any, Any], bool]] = {
    "eq": operator.eq,
    "ne": operator.ne,
    "gt": operator.gt,
    "ge": operator.ge,
    "lt": operator.lt,
    "le": operator.le,
}


def compile_rule(rule: dict[str, Any]) -> Callable[[Any], bool]:
    required = {"field", "op", "value"}
    missing = required - rule.keys()
    if missing:
        raise ValueError(f"rule is missing keys: {sorted(missing)}")

    field = rule["field"]
    operation_name = rule["op"]
    expected = rule["value"]
    if operation_name not in OPERATORS:
        raise ValueError(f"unsupported operation: {operation_name!r}")

    comparison = OPERATORS[operation_name]

    def predicate(item: Any) -> bool:
        actual = getattr(item, field)
        return False if actual is None else comparison(actual, expected)

    return predicate


def compile_rules(rules: Sequence[dict[str, Any]]) -> Callable[[Any], bool]:
    predicates = [compile_rule(rule) for rule in rules]
    return lambda item: all(predicate(item) for predicate in predicates)


rules = [
    {"field": "origin", "op": "eq", "value": "Japan"},
    {"field": "mpg", "op": "ge", "value": 28},
    {"field": "model_year", "op": "ge", "value": 76},
]

configured = list(
    Pipeline(iter_cars(DATA_FILE))
    | filtering(compile_rules(rules))
    | mapping(lambda car: (car.name, car.mpg, car.model_year))
)

configured

[('Honda Civic', 33.0, 76),
 ('Datsun B-210', 32.0, 76),
 ('Toyota Corolla', 28.0, 76),
 ('Honda Accord CVCC', 31.5, 77),
 ('Datsun F-10 Hatchback', 33.5, 77),
 ('Subaru DL', 30.0, 77),
 ('Mazda GLC Deluxe', 32.8, 78),
 ('Datsun B210 GX', 39.4, 78),
 ('Honda Civic CVCC', 36.1, 78),
 ('Honda Accord LX', 29.5, 78),
 ('Mazda GLC Deluxe', 34.1, 79),
 ('Datsun 210', 31.8, 79),
 ('Toyota Corolla Tercel', 38.1, 80),
 ('Datsun 310', 37.2, 80),
 ('Toyota Corolla Liftback', 29.8, 80),
 ('Mazda 626', 31.3, 80),
 ('Datsun 510 Hatchback', 37.0, 80),
 ('Toyota Corolla', 32.2, 80),
 ('Mazda GLC', 46.6, 80),
 ('Datsun 210', 40.8, 80),
 ('Honda Civic 1500 gl', 44.6, 80),
 ('Subaru DL', 33.8, 80),
 ('Datsun 280-ZX', 32.7, 80),
 ('Honda Accord', 32.4, 80),
 ('Toyota Starlet', 39.1, 81),
 ('Honda Civic 1300', 35.1, 81),
 ('Subaru', 32.3, 81),
 ('Datsun 210 MPG', 37.0, 81),
 ('Toyota Tercel', 37.7, 81),
 ('Mazda GLC 4', 34.1, 81),
 ('Honda Prelude', 33.7, 81),
 ('Toyota Corolla', 32.4, 81),
 ('Datsun 200SX

## Problem 16 — Group a sorted stream

`itertools.groupby` groups only adjacent equal keys. Sort the demonstration data by `(origin, model_year)`, then calculate count and average MPG per group.

For large data, use an external sort or push ordering into a database query.

In [21]:
def summarize_group(key: tuple[str, int], cars: Iterable[Car]):
    count = 0
    mpg_total = 0.0
    mpg_count = 0
    for car in cars:
        count += 1
        if car.mpg is not None:
            mpg_total += car.mpg
            mpg_count += 1
    return {
        "origin": key[0],
        "model_year": key[1],
        "count": count,
        "average_mpg": mpg_total / mpg_count if mpg_count else None,
    }


sorted_cars = iter(
    sorted(iter_cars(DATA_FILE), key=lambda car: (car.origin, car.model_year))
)

group_summaries = (
    summarize_group(key, group)
    for key, group in itertools.groupby(
        sorted_cars,
        key=lambda car: (car.origin, car.model_year),
    )
)

for summary in itertools.islice(group_summaries, 10):
    print(summary)

{'origin': 'Europe', 'model_year': 70, 'count': 6, 'average_mpg': 21.0}
{'origin': 'Europe', 'model_year': 71, 'count': 5, 'average_mpg': 23.0}
{'origin': 'Europe', 'model_year': 72, 'count': 5, 'average_mpg': 22.0}
{'origin': 'Europe', 'model_year': 73, 'count': 7, 'average_mpg': 24.0}
{'origin': 'Europe', 'model_year': 74, 'count': 6, 'average_mpg': 27.0}
{'origin': 'Europe', 'model_year': 75, 'count': 6, 'average_mpg': 24.5}
{'origin': 'Europe', 'model_year': 76, 'count': 8, 'average_mpg': 24.25}
{'origin': 'Europe', 'model_year': 77, 'count': 4, 'average_mpg': 29.25}
{'origin': 'Europe', 'model_year': 78, 'count': 6, 'average_mpg': 24.95}
{'origin': 'Europe', 'model_year': 79, 'count': 4, 'average_mpg': 30.45}


## Problem 17 — Implement a generic CSV sink

Consume dictionaries lazily, infer field names from the first record, write a header, return the number of rows written, and handle empty inputs.

In [22]:
def write_dict_rows(
    file_name: str | Path,
    rows: Iterable[dict[str, Any]],
    *,
    encoding: str = "utf-8",
) -> int:
    iterator = iter(rows)
    try:
        first_row = next(iterator)
    except StopIteration:
        Path(file_name).write_text("", encoding=encoding)
        return 0

    fieldnames = list(first_row.keys())
    count = 0
    with Path(file_name).open("w", encoding=encoding, newline="") as file_obj:
        writer = csv.DictWriter(file_obj, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerow(first_row)
        count += 1
        for row in iterator:
            writer.writerow(row)
            count += 1
    return count


export_rows = (
    asdict(car)
    for car in iter_cars(DATA_FILE)
    if car.mpg is not None and car.mpg >= 30
)

with tempfile.TemporaryDirectory() as temp_dir:
    export_file = Path(temp_dir) / "high_mpg.csv"
    written = write_dict_rows(export_file, export_rows)
    print("rows written:", written)
    print(export_file.read_text(encoding="utf-8"))

rows written: 92
name,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin
Peugeot 304,30.0,4,79.0,70.0,2074.0,19.5,71,Europe
Fiat 124B,30.0,4,88.0,76.0,2065.0,14.5,71,Europe
Toyota Corolla 1200,31.0,4,71.0,65.0,1773.0,19.0,71,Japan
Datsun 1200,35.0,4,72.0,69.0,1613.0,18.0,71,Japan
Datsun B210,31.0,4,79.0,67.0,1950.0,19.0,74,Japan
Toyota Corolla 1200,32.0,4,71.0,65.0,1836.0,21.0,74,Japan
Toyota Corolla,31.0,4,76.0,52.0,1649.0,16.5,74,Japan
Datsun 710,32.0,4,83.0,61.0,2003.0,19.0,74,Japan
Fiat x1.9,31.0,4,79.0,67.0,2000.0,16.0,74,Europe
Honda Civic CVCC,33.0,4,91.0,53.0,1795.0,17.5,75,Japan
Honda Civic,33.0,4,91.0,53.0,1795.0,17.4,76,Japan
Datsun B-210,32.0,4,85.0,70.0,1990.0,17.0,76,Japan
Honda Accord CVCC,31.5,4,98.0,68.0,2045.0,18.5,77,Japan
Buick Opel Isuzu Deluxe,30.0,4,111.0,80.0,2155.0,14.8,77,US
Renault 5 GTL,36.0,4,79.0,58.0,1825.0,18.6,77,Europe
Datsun F-10 Hatchback,33.5,4,85.0,70.0,1945.0,16.8,77,Japan
Chevrolet Chevette,30.5,4,98.0,63.0,2051.0,17.0,77

## Problem 18 — End-to-end advanced pipeline

Build one pipeline that:

1. reads and parses cars lazily;
2. records source metrics;
3. keeps model year 1975 or later;
4. requires known MPG and horsepower;
5. computes horsepower-to-weight;
6. keeps MPG at least 25;
7. returns the top five by horsepower-to-weight;
8. produces serialization-ready dictionaries.

In [23]:
@dataclass(frozen=True, slots=True)
class CarPerformance:
    name: str
    origin: str
    model_year: int
    mpg: float
    horsepower: float
    weight: float
    power_to_weight: float


def to_performance(car: Car) -> CarPerformance:
    if car.mpg is None or car.horsepower is None:
        raise ValueError("MPG and horsepower are required")
    return CarPerformance(
        name=car.name,
        origin=car.origin,
        model_year=car.model_year,
        mpg=car.mpg,
        horsepower=car.horsepower,
        weight=car.weight,
        power_to_weight=car.horsepower / car.weight,
    )


final_metrics = PipelineMetrics()
eligible = (
    Pipeline(iter_cars(DATA_FILE))
    | inspecting(final_metrics.observe)
    | filtering(lambda car: car.model_year >= 75)
    | filtering(lambda car: car.mpg is not None)
    | filtering(lambda car: car.horsepower is not None)
    | mapping(to_performance)
    | filtering(lambda item: item.mpg >= 25)
)

top_performance = top_k(
    eligible,
    5,
    key=lambda item: item.power_to_weight,
)

final_output = [asdict(item) for item in top_performance]
print("source metrics:", final_metrics)
for row in final_output:
    print(row)

source metrics: PipelineMetrics(total=406, missing_mpg=0, by_origin=defaultdict(<class 'int'>, {'US': 254, 'Europe': 73, 'Japan': 79}))
{'name': 'Datsun 280-ZX', 'origin': 'Japan', 'model_year': 80, 'mpg': 32.7, 'horsepower': 132.0, 'weight': 2910.0, 'power_to_weight': 0.04536082474226804}
{'name': 'Chevrolet Citation', 'origin': 'US', 'model_year': 79, 'mpg': 28.8, 'horsepower': 115.0, 'weight': 2595.0, 'power_to_weight': 0.04431599229287091}
{'name': 'Saab 99LE', 'origin': 'Europe', 'model_year': 75, 'mpg': 25.0, 'horsepower': 115.0, 'weight': 2671.0, 'power_to_weight': 0.0430550355672033}
{'name': 'Oldsmobile Omega Brougham', 'origin': 'US', 'model_year': 79, 'mpg': 26.8, 'horsepower': 115.0, 'weight': 2700.0, 'power_to_weight': 0.04259259259259259}
{'name': 'Datsun 510', 'origin': 'Japan', 'model_year': 78, 'mpg': 27.2, 'horsepower': 97.0, 'weight': 2300.0, 'power_to_weight': 0.04217391304347826}


# Tests

Strong tests verify laziness, boundary cases, identity preservation, allowed operations, and numerical correctness—not only final output.

In [24]:
def test_take_is_lazy() -> None:
    produced: list[int] = []

    def source() -> Iterator[int]:
        for value in range(10):
            produced.append(value)
            yield value

    assert list(take(3, source())) == [0, 1, 2]
    assert produced == [0, 1, 2]


def test_filter_preserves_identity() -> None:
    objects = [{"value": 1}, {"value": 2}]
    result = list(filter_items(lambda item: item["value"] == 2, objects))
    assert result[0] is objects[1]


def test_batched_boundaries() -> None:
    assert list(batched([], 3)) == []
    assert list(batched([1], 3)) == [(1,)]
    assert list(batched([1, 2, 3], 3)) == [(1, 2, 3)]
    assert list(batched([1, 2, 3, 4], 3)) == [(1, 2, 3), (4,)]


def test_sliding_window() -> None:
    assert list(sliding_window([1, 2, 3, 4], 3)) == [
        (1, 2, 3),
        (2, 3, 4),
    ]


def test_pipeline_operator() -> None:
    result = (
        Pipeline(range(10))
        | filtering(lambda value: value % 2 == 0)
        | mapping(lambda value: value * 10)
        | taking(3)
    ).collect()
    assert result == [0, 20, 40]


def test_unknown_rule_operation_is_rejected() -> None:
    try:
        compile_rule({"field": "mpg", "op": "execute", "value": 1})
    except ValueError as exc:
        assert "unsupported operation" in str(exc)
    else:
        raise AssertionError("Expected ValueError")


def test_online_stats() -> None:
    stats = OnlineStats()
    values = [1.0, 2.0, 3.0, 4.0]
    for value in values:
        stats.update(value)
    assert stats.count == 4
    assert math.isclose(stats.mean, 2.5)
    assert math.isclose(stats.sample_variance, statistics.variance(values))


tests = [
    test_take_is_lazy,
    test_filter_preserves_identity,
    test_batched_boundaries,
    test_sliding_window,
    test_pipeline_operator,
    test_unknown_rule_operation_is_rejected,
    test_online_stats,
]

for test in tests:
    test()

print(f"{len(tests)} tests passed.")

7 tests passed.


# Additional advanced practice

## A. Timeout-aware source

Pull pages from a slow API until a time budget is exhausted. Inject the clock so tests do not depend on wall-clock time.

## B. Retry transient failures

Retry only an explicit tuple of transient exception types. Use capped exponential backoff and never retry validation errors.

## C. Merge sorted streams

Use `heapq.merge(..., key=...)` to merge several already-sorted sources lazily.

## D. Exact duplicate detection

Track `(name, model_year)` in a set. Explain why exact detection needs memory proportional to the number of distinct keys.

## E. Approximate duplicate detection

Replace the set with a Bloom filter. Analyze false positives, capacity, and error rate.

## F. Asynchronous pull pipelines

Rebuild the source and stages with `async def`, `yield`, and `async for`. Make cancellation and cleanup explicit.

## G. Pull versus push backpressure

Compare this iterator design with an `asyncio.Queue` pipeline. Identify who controls production and where buffering occurs.

# Best-practice checklist

- Prefer named columns over numeric indexes.
- Convert strings into typed records near the source.
- Separate source, parsing, validation, transformation, filtering, aggregation, and sink concerns.
- Make the error policy explicit: fail, skip, substitute, or dead-letter.
- Keep stages lazy unless materialization is required.
- Use bounded-memory tools: `islice`, fixed `deque`, batches, heaps, and online algorithms.
- Document sorted-input requirements.
- Remember that iterators are usually single-pass.
- Use `tee` carefully because uneven consumers create buffers.
- Never compile configuration with `eval`.
- Inject clocks, clients, and loggers for testability.
- Test laziness, cleanup, and boundary cases.
- Keep side effects in clearly named inspection or sink stages.

# Summary

Pull pipelines let the final consumer control demand. Generators make them concise, while robust designs require deliberate choices about typing, validation, error handling, resource lifetime, bounded memory, observability, branching, and testing.

The same architecture applies to files, database cursors, paginated APIs, message streams, and asynchronous data sources.